<a id="parsing"></a>
# Parsing de CorBaMa — génération de corbama_corpus.tsv

Cette section transforme les fichiers HTML annotés de CorBaMa en un fichier CoNLL/TSV, exactement au même format que bambara_pos_prep_retagged.conll (module 04), pour pouvoir réutiliser directement load_bambara_corpus et tous les outils déjà construits. corbama_corpus.tsv est entièrement généré par les cellules ci-dessous, rien à écrire à la main.

In [1]:
import sys, glob
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

html_files = glob.glob(str(Path.cwd() / "corbama-disamb" / "**" / "*.dis.html"), recursive=True)
print(f"{len(html_files)} fichiers HTML trouvés.")

2605 fichiers HTML trouvés.


In [2]:
all_sentences = []
errors = []
for fp in html_files:
    try:
        all_sentences.extend(parse_corbama_file(fp))
    except Exception as e:
        errors.append((fp, str(e)))

print(f"Phrases extraites : {len(all_sentences)}")
print(f"Tokens extraits (mots + ponctuation) : {sum(len(w) for w, _, _ in all_sentences)}")
print(f"Erreurs de parsing : {len(errors)}")
for fp, e in errors:
    print("  ", fp, "->", e)

Phrases extraites : 166361
Tokens extraits (mots + ponctuation) : 2631942
Erreurs de parsing : 0


In [3]:
# Statistiques : distribution des tags mappés (11 classes, comparable à Bayelemabaga)
from collections import Counter

mapped_counter = Counter(t for _, _, mt in all_sentences for t in mt)
total = sum(mapped_counter.values())

print("Distribution des tags (tagset mappé) :")
for tag, count in mapped_counter.most_common():
    print(f"  {tag:<8} {count:>6} ({100*count/total:.1f}%)")

Distribution des tags (tagset mappé) :
  NOM      635374 (24.1%)
  PUNCT    389376 (14.8%)
  PRON     361139 (13.7%)
  AUX      341048 (13.0%)
  VERBE    323993 (12.3%)
  POSTP    240354 (9.1%)
  CONJ     101866 (3.9%)
  ADJ       80453 (3.1%)
  DET       79879 (3.0%)
  PART      55415 (2.1%)
  ADV       23045 (0.9%)


## Vérification manuelle avant export

Toujours vérifier visuellement quelques phrases avant de faire confiance à un parseur — c'est ce qui a permis de trouver les deux bugs (ponctuation ignorée, artefact BOM) avant de livrer ce code.

In [4]:
words, fine_tags, mapped_tags = all_sentences[0]
print("Phrase reconstituée :", " ".join(words))
print()
for w, ft, mt in zip(words, fine_tags, mapped_tags):
    print(f"  {w:<15} fin={ft:<10} mappé={mt}")

Phrase reconstituée : bana minnu ka teli ka mògò minè Mali duguw kònò

  bana            fin=n          mappé=NOM
  minnu           fin=dtm        mappé=DET
  ka              fin=pm         mappé=AUX
  teli            fin=vq         mappé=VERBE
  ka              fin=pm         mappé=AUX
  mògò          fin=n          mappé=NOM
  minè           fin=v          mappé=VERBE
  Mali            fin=n.prop     mappé=NOM
  duguw           fin=n          mappé=NOM
  kònò          fin=pp         mappé=POSTP


In [5]:
# Export : deux fichiers, tagset mappé (11 classes, pour les modules 05/06/07) et tagset fin (24 classes, pour référence)
export_corbama_to_conll(all_sentences, "corbama_corpus.tsv", tag_level="mapped")
export_corbama_to_conll(all_sentences, "corbama_corpus_fine.tsv", tag_level="fine")

print("Fichiers générés :")
print(" -", Path("corbama_corpus.tsv").resolve(), f"({Path('corbama_corpus.tsv').stat().st_size} octets)")
print(" -", Path("corbama_corpus_fine.tsv").resolve(), f"({Path('corbama_corpus_fine.tsv').stat().st_size} octets)")

Fichiers générés :
 - /home/lex_luthor/Documents/A-GENRAL/bambara-pos-tagging/08_real_data_evaluation/corbama_corpus.tsv (25278469 octets)
 - /home/lex_luthor/Documents/A-GENRAL/bambara-pos-tagging/08_real_data_evaluation/corbama_corpus_fine.tsv (22161107 octets)


In [6]:
# Vérification finale : le fichier généré se relit correctement avec l'outil du projet
reloaded = load_bambara_corpus("corbama_corpus.tsv")
assert len(reloaded) == len(all_sentences), "Incohérence entre l'export et le rechargement !"
print(f"Rechargement cohérent : {len(reloaded)} phrases.")

Corpus chargé : 166361 phrases.
Rechargement cohérent : 166361 phrases.
